# Visualize NWM



## Load model


In [3]:
import yaml
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
from PIL import Image
import requests
from io import BytesIO
import numpy as np
from IPython.display import display, HTML
import ipywidgets as widgets
from diffusers.models import AutoencoderKL

from diffusion import create_diffusion
from isolated_nwm_infer import model_forward_wrapper
from misc import transform
from models import CDiT_models
from datasets import TrainingDataset

EXP_NAME = 'nwm_cdit_xl'
# MODEL_PATH = f'logs/{EXP_NAME}/checkpoints/0100000.pth.tar'

MODEL_PATH="../cdit_s_100000.pth.tar"

with open("config/data_config.yaml", "r") as f:
    default_config = yaml.safe_load(f)
config = default_config

with open(f'config/{EXP_NAME}.yaml', "r") as f:
    user_config = yaml.safe_load(f)
config.update(user_config)
latent_size = config['image_size'] // 8

print("loading model")
model = CDiT_models['CDiT-S/2'](input_size=latent_size, context_size=config['context_size'])
#model = CDiT_models[](input_size=latent_size, context_size=config['context_size'])
ckp = torch.load(MODEL_PATH, map_location='cpu', weights_only=False) 

print(model.load_state_dict(ckp["ema"], strict=True))
model.eval()
device = 'cuda:0'
model.to(device)
model = torch.compile(model)

diffusion = create_diffusion(str(250))
vae = AutoencoderKL.from_pretrained(f"stabilityai/sd-vae-ft-ema").to(device)
latent_size = config['image_size'] // 8



loading model
<All keys matched successfully>


Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.
/venv/main/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

In [ ]:

import os
import torch
import onnxruntime as ort
import numpy as np

# Create directory to save ONNX models explicitly
onnx_dir = "./onnx_models"
os.makedirs(onnx_dir, exist_ok=True)
cdit_p, enc_p, dec_p = f"{onnx_dir}/cdit.onnx", f"{onnx_dir}/enc.onnx", f"{onnx_dir}/dec.onnx"

print("Exporting models to ONNX... (This might take a few minutes)")

class CDiTWrap(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x, t, y, x_cond, rel_t): return self.m(x, t, y, x_cond, rel_t)

class EncWrap(torch.nn.Module):
    def __init__(self, v): super().__init__(); self.v = v
    def forward(self, px): return self.v.encode(px).latent_dist.mean

class DecWrap(torch.nn.Module):
    def __init__(self, v): super().__init__(); self.v = v
    def forward(self, z): return self.v.decode(z / 0.18215).sample
    
with torch.no_grad():
    model_fp32 = model.float()
    vae_fp32 = vae.float()
    ls = latent_size
    cs = config["context_size"]

    if not os.path.exists(cdit_p):
        print("Exporting CDiT...")
        torch.onnx.export(CDiTWrap(model_fp32).eval(),
            (torch.randn(1,4,ls,ls,device=device,dtype=torch.float32), torch.tensor([100.],device=device,dtype=torch.float32),
             torch.randn(1,3,device=device,dtype=torch.float32), torch.randn(1,cs,4,ls,ls,device=device,dtype=torch.float32),
             torch.tensor([0.0078125],device=device,dtype=torch.float32)), cdit_p, export_params=True, opset_version=18, do_constant_folding=True,
            input_names=["x","t","y","x_cond","rel_t"], output_names=["out"])
    
    if not os.path.exists(enc_p):
        print("Exporting VAE Encoder...")
        torch.onnx.export(EncWrap(vae_fp32).eval(), (torch.randn(1,3,224,224,device=device,dtype=torch.float32),),
            enc_p, export_params=True, opset_version=18, do_constant_folding=True, input_names=["px"], output_names=["lat"])
            
    if not os.path.exists(dec_p):
        print("Exporting VAE Decoder...")
        torch.onnx.export(DecWrap(vae_fp32).eval(), (torch.randn(1,4,ls,ls,device=device,dtype=torch.float32),),
            dec_p, export_params=True, opset_version=18, do_constant_folding=True, input_names=["z"], output_names=["px"])

print("ONNX Models exported successfully! Loading ONNX Sessions...")

trt_cache = "./trt_cache"
os.makedirs(trt_cache, exist_ok=True)
providers = [
    ("TensorrtExecutionProvider", {"trt_bf16_enable": True, "trt_engine_cache_enable": True, "trt_engine_cache_path": trt_cache, "trt_max_workspace_size": 2147483648}), 
    "CUDAExecutionProvider", 
    "CPUExecutionProvider"
]
enc_sess = ort.InferenceSession(enc_p, providers=providers)
cdit_sess = ort.InferenceSession(cdit_p, providers=providers)
dec_sess = ort.InferenceSession(dec_p, providers=providers)

def ort_run(session, inputs): return session.run([session.get_outputs()[0].name], inputs)[0]

print("Overriding model_forward_wrapper for TensorRT Inference...")

# Create overriding forward wrapper
def onnx_model_forward_wrapper(all_models, curr_obs, curr_delta, num_timesteps, latent_sz, dev, num_cond, num_goals=1, rel_t=None, progress=False):
    _, diff_obj, _ = all_models
    x = curr_obs.to(dev)
    y = curr_delta.to(dev)
    B, T = x.shape[:2]

    if rel_t is None:
        rel_t = (torch.ones(B)* (1. / 128.)).to(dev)
        rel_t *= num_timesteps

    # Encode Context (VAE)  -- using ONNX
    x_np = x.flatten(0,1).cpu().numpy().astype(np.float32)
    lat_list = [ort_run(enc_sess, {"px": x_np[i:i+1]}) for i in range(x_np.shape[0])]
    x_enc = np.concatenate(lat_list, axis=0) * 0.18215
    x_enc = x_enc.reshape(B, T, 4, latent_sz, latent_sz)
    
    # Condition Variables
    x_cond_np = x_enc[:, :num_cond]
    x_cond_np = np.repeat(x_cond_np[:, np.newaxis, ...], num_goals, axis=1) # expand for goals
    x_cond_np = x_cond_np.reshape(B * num_goals, num_cond, 4, latent_sz, latent_sz)
    
    y_np = y.flatten(0, 1).cpu().numpy().astype(np.float32)
    rel_t_np = rel_t.cpu().numpy().astype(np.float32)
    z = torch.randn(B*num_goals, 4, latent_sz, latent_sz, device=dev)
    
    # CDiT Denoising Function -- using ONNX TRT
    def onnx_model_fn(x_in, t_in, y_in, x_cond_in, rel_t_in):
        out = ort_run(cdit_sess, {
            "x": x_in.cpu().numpy().astype(np.float32),
            "t": t_in.cpu().numpy().astype(np.float32),
            "y": y_in.cpu().numpy().astype(np.float32),
            "x_cond": x_cond_in.cpu().numpy().astype(np.float32),
            "rel_t": rel_t_in.cpu().numpy().astype(np.float32)
        })
        return torch.from_numpy(out).to(dev)

    x_cond_t = torch.from_numpy(x_cond_np).to(dev)
    model_kwargs = dict(y=y, x_cond=x_cond_t, rel_t=rel_t)      
    
    # Diffusion Loop
    samples = diff_obj.p_sample_loop(
        lambda x_val, t_val, **kw: onnx_model_fn(x_val, t_val, kw["y"], kw["x_cond"], kw["rel_t"]),
        z.shape, z, clip_denoised=False, model_kwargs=model_kwargs, progress=progress, device=dev
    )
    
    # Decode VAE -- using ONNX
    samples_np = (samples.cpu().numpy().astype(np.float32) / 0.18215)
    dec_list = [ort_run(dec_sess, {"z": samples_np[i:i+1]}) for i in range(samples_np.shape[0])]
    decoded = np.concatenate(dec_list, axis=0)
    
    return torch.clip(torch.from_numpy(decoded).to(dev), -1., 1.)

# Apply the global hook directly inside this notebook instance!
model_forward_wrapper = onnx_model_forward_wrapper
print("Successfully overrode `model_forward_wrapper` with ONNX TensorRT bfloat16 optimizations!")


## Choose starting image

In [4]:
def url_to_pil_image(url):
    response = requests.get(url)
    img = Image.open(BytesIO(response.content)).convert('RGB')
    return img

def load_internet_image(url):
    from torchvision import transforms
    _transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5], inplace=True),
    ])
    img = url_to_pil_image(url)
    x_start = _transform(img)
    return x_start.unsqueeze(0).expand(config['context_size'], x_start.shape[0], x_start.shape[1], x_start.shape[2])
    
# Jupyter Notebook Cell


# List of image links
# For unknown environments, please see https://github.com/facebookresearch/nwm/issues/7
image_links = [
    'https://raw.githubusercontent.com/amirbar/amirbar.github.io/refs/heads/master/images/recon.png',
    'https://raw.githubusercontent.com/amirbar/amirbar.github.io/refs/heads/master/images/scand.png',
    'https://raw.githubusercontent.com/amirbar/amirbar.github.io/refs/heads/master/images/sacson.png',
    'https://raw.githubusercontent.com/amirbar/amirbar.github.io/refs/heads/master/images/tartan.png'
]

# Output widget to hold the selected link
output = widgets.Output()
x_start_link = None  # This will hold the selected link

# Function to handle image click
def on_image_click(link):
    global x_start_link
    x_start_link = link
    with output:
        output.clear_output()
        print(f"Selected image link:\n{x_start_link}")

# Create HBox of images
image_buttons = []
for link in image_links:
    img = widgets.Button(
        description='click',
        layout=widgets.Layout(width='150px', height='20px', padding='0'),
        style={'button_color': 'lightgray'}
    )

    img._dom_classes += ('image-button',)
    img_link = link  # capture current link in closure

    def on_click(b, link=img_link):
        on_image_click(link)

    img.on_click(on_click)

    # Embed image using HTML style
    img_html = f'<img src="{link}" width="150px" height="150px">'
    img_html_widget = widgets.HTML(value=img_html)
    image_buttons.append(widgets.VBox([img_html_widget, img]))

# Display the gallery
display(widgets.HBox(image_buttons))
display(output)


Output()

## Visualize navigation commands

In [ ]:
x_start = load_internet_image(x_start_link)

commands = {
    'Forward': [1,0,0],
    'Rotate Right': [0,0,-0.5],
    'Rotate Left': [0,0,0.5],
}
preds = {}

def reset():
    x_cond_pixels = x_start
    reconstructed_image=x_cond_pixels.to(device)
    preds['x_cond_pixels_display'] = (reconstructed_image[-1] * 127.5 + 127.5).clamp(0, 255).permute(1, 2, 0).to("cpu", dtype=torch.uint8).numpy()
    preds['x_cond_pixels'] = x_cond_pixels
    preds['video'] = [preds['x_cond_pixels_display']]
    

reset()
display(Image.fromarray(preds['x_cond_pixels_display']))
Image.fromarray(preds['x_cond_pixels_display']).save('sacson.png')

output = widgets.Output()
display(output)
rel_t = (torch.ones(1)*0.0078125).to(device)

@output.capture()
def update_image(b):
    
    if b.description == 'Reset':
        print("Reset clicked!")
        output.clear_output(wait=False)
        reset()
        return

    print("Button clicked!")
    y = commands[b.description]
    y = torch.tensor(y).to(device).unsqueeze(0)
    
    print("You entered:", b.description)
    x_cond_pixels = preds['x_cond_pixels'][-4:].unsqueeze(0).to(device)
    samples = model_forward_wrapper((model, diffusion, vae), x_cond_pixels, y, None, latent_size, device, config["context_size"], num_goals=1, rel_t=rel_t, progress=True)
    x_cond_pixels = samples # torch.clip(samples, -1., 1.)
    preds['x_cond_pixels'] = torch.cat([preds['x_cond_pixels'].to(x_cond_pixels), x_cond_pixels], dim=0)
    samples = (samples * 127.5 + 127.5).permute(0, 2, 3, 1).clamp(0,255).to("cpu", dtype=torch.uint8).numpy()
    display(Image.fromarray(samples[0]))
    preds['video'].append(samples[0])

buttons = []
for o in ["Forward", "Rotate Left", "Rotate Right", "Reset"]:
    b = widgets.Button(description=o)
    b.on_click(update_image)
    display(b)
    buttons.append(b)

# Generate a video

In [ ]:

%matplotlib inline
from matplotlib import pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import numpy as np
# np array with shape (frames, height, width, channels)
video = np.array(preds['video']) 

fig = plt.figure()
im = plt.imshow(video[0,:,:,:])

plt.close() # this is required to not display the generated image

def init():
    im.set_data(video[0,:,:,:])

def animate(i):
    im.set_data(video[i,:,:,:])
    return im

anim = animation.FuncAnimation(fig, animate, init_func=init, frames=video.shape[0],
                               interval=500)
HTML(anim.to_html5_video())


In [ ]:
# optional: load from dataset
# dataloaders = {}

# for dataset_name in config["datasets"]:
#     data_config = config["datasets"][dataset_name]
#     for data_split_type in ["test"]: #["test"]:
#         dataset = TrainingDataset(
#             data_folder=data_config["data_folder"],
#             data_split_folder=data_config[data_split_type],
#             dataset_name=dataset_name,
#             image_size=config["image_size"],
#             min_dist_cat=config["distance"]["min_dist_cat"],
#             max_dist_cat=config["distance"]["max_dist_cat"],
#             len_traj_pred=config["len_traj_pred"],
#             context_size=config["context_size"],
#             normalize=config["normalize"],
#             goals_per_obs=1,
#             transform=transform,
#             predefined_index=None,
#             traj_stride=1,
#         )
#         dataloaders[f"{dataset_name}_{data_split_type}"] = dataset
#         print(f"Dataset: {dataset_name} ({data_split_type}), size: {len(dataset)}")

# load from dataset
# ds = dataloaders['recon_test'] # scand_test, 
# x, _, _ = ds[np.random.randint(len(ds))]
# x_start = x[:config["context_size"]]
